# UDA-Hub Agentic App

Entrypoint for the compiled multi-agent ticket-resolution graph. See
`agentic/design/architecture.md` for the full architecture writeup, and
`agentic/workflow.py` for the hand-built `StateGraph` this notebook drives
(no `create_react_agent` / `langgraph_supervisor` prebuilt helpers used).

**How to run**: install `solution/requirements.txt` into the project venv,
put a real `OPENAI_API_KEY` in `solution/.env` (see `.env` for the
variable name expected by `load_dotenv()` below -- never commit this
file), then run all cells top to bottom. The `chat_interface()` call opens
an interactive input loop in the cell output; type `quit`/`exit`/`q` to end
the session.

## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

Agents live under `agentic/agents/`, tools under `agentic/tools/`, and the
orchestration graph is assembled in `agentic/workflow.py`.

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

### Demo identity

`chat_interface()` seeds each turn with the ticket/account/user identity a
real support channel would supply, since `context_loader_node` needs it to
look up the customer's CultPass profile, subscription, ticket history, and
long-term memories before any agent reasons about the ticket. The values
below match a customer already seeded by `02_core_db_setup.ipynb` into
`data/core/udahub.db` / `data/external/cultpass.db` (account `cultpass`,
external user `a4ab87`); swap in any other seeded `external_user_id` from
those databases to try a different customer. `ticket_id` also doubles as
the LangGraph `thread_id`, so re-running with the same id resumes the same
short-term (session) memory; use a fresh id to start a new ticket.

In [ ]:
chat_interface(
    orchestrator,
    ticket_id="demo-1",
    account_id="cultpass",
    external_user_id="a4ab87",
    channel="chat",
)

User: 
Assistant: Thank you for your patience. I have escalated your request to a specialist who will be able to assist you further with your account issue.
User: 
Assistant: Thank you for your patience. I want to let you know that your request has been passed to a specialist who will assist you further with your account issue.
User: 
Assistant: Thank you for your patience. I have escalated your request to a specialist who will be able to assist you further with your account issue.
User: 
Assistant: Thank you for your patience. I want to let you know that your request has been passed to a specialist who will assist you further with your account issue.


### Inspect session state

The compiled graph's checkpointer (`MemorySaver`, keyed by `thread_id`)
retains the full message history and last-computed state for the ticket
session above -- this is the short-term/session memory layer described in
`agentic/design/architecture.md`.

In [ ]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "demo-1",
        }
    }
))[0].values["messages"]